In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

SHEET_ID = os.getenv('SHEET_ID')

print('Sheet ID loaded:', bool(SHEET_ID))


In [ ]:
import os

import gspread

SHEET_KEY = os.getenv("SHEET_ID")
SERVICE_ACCOUNT = os.getenv("SERVICE_ACCOUNT_PATH", "service_account.json")

class LeaveSheet:
    def __init__(self, key=SHEET_KEY, cred=SERVICE_ACCOUNT):
        gc = gspread.service_account(filename=cred)
        self.ws = gc.open_by_key(key).sheet1
        self.reload()

    def reload(self):
        """Pull all values once, build employee->column map."""
        self.rows = self.ws.get_all_values()   # list[list[str]], 0-indexed
        id_row   = self.rows[0]
        name_row = self.rows[1]
        head_row = self.rows[2]

        # block starts = columns where an ID exists (skip col 0)
        starts = [i for i in range(1, len(id_row)) if id_row[i].strip()]
        starts.append(len(id_row))  # sentinel end

        self.employees = {}   # name -> meta
        for b in range(len(starts) - 1):
            c0, c1 = starts[b], starts[b + 1]
            emp_id = id_row[c0].strip()
            name   = name_row[c0].strip()
            # map sub-header -> absolute col index within this block
            fields = {}
            for c in range(c0, c1):
                key = head_row[c].strip().lower()  # credit / lwp / debit
                if key:
                    fields[key] = c
            self.employees[name] = {
                "id": emp_id,
                "col_start": c0,
                "fields": fields,   # {'credit': idx, 'lwp': idx, 'debit': idx}
            }

        # date rows: row index 3.. until first blank date cell
        self.date_rows = {}   # date string -> 0-indexed row
        r = 3
        while r < len(self.rows) and self.rows[r][0].strip():
            self.date_rows[self.rows[r][0].strip()] = r
            r += 1
        self._summary_start = r  # blank line after data

    @staticmethod
    def _num(val):
        """Parse cell to float; keep raw string if it's a note like '(1-26Jan)'."""
        v = val.strip()
        if v in ("", "-"):
            return 0.0
        try:
            return float(v)
        except ValueError:
            return v   # annotation, keep raw

    # ---- READ ----
    def list_employees(self):
        return list(self.employees.keys())

    def get_all_data(self):
        """Everything: employee -> date -> {credit,lwp,debit}."""
        out = {}
        for name, meta in self.employees.items():
            out[name] = self.get_employee(name)
        return out

    def get_employee(self, name):
        meta = self.employees[name]
        data = {"id": meta["id"], "leaves": {}}
        for date, r in self.date_rows.items():
            row = self.rows[r]
            rec = {}
            for f, c in meta["fields"].items():
                raw = row[c] if c < len(row) else ""
                rec[f] = self._num(raw)
            data["leaves"][date] = rec
        return data

    def get_balance(self, name):
        """Read 'Leave Balance' row value for employee."""
        meta = self.employees[name]
        for r in range(self._summary_start, len(self.rows)):
            if self.rows[r] and self.rows[r][0].strip().lower() == "leave balance":
                c = meta["col_start"]
                return self._num(self.rows[r][c]) if c < len(self.rows[r]) else 0.0
        return None


    # ---- WRITE ----
    def set_value(self, name, date, field, value):
        """field = 'credit' | 'lwp' | 'debit'. Writes one cell."""
        meta = self.employees[name]
        if field.lower() not in meta["fields"]:
            raise ValueError(f"{name} has no '{field}' column")
        if date not in self.date_rows:
            raise ValueError(f"date '{date}' not found")
        r = self.date_rows[date]            # 0-indexed
        c = meta["fields"][field.lower()]   # 0-indexed
        self.ws.update_cell(r + 1, c + 1, value)   # gspread is 1-indexed
        self.rows[r][c] = str(value)               # keep cache in sync

    def add_leave(self, name, date, days, field="debit"):
        """Add `days` to existing value (e.g. employee took leave)."""
        cur = self.get_employee(name)["leaves"].get(date, {}).get(field, 0.0)
        if not isinstance(cur, (int, float)):
            raise ValueError(f"cell has note '{cur}', fix manually")
        self.set_value(name, date, field, cur + days)

In [21]:
s = LeaveSheet()
print(s.list_employees())                      # all names
# s.get_all_data()                         # full dump
print(s.get_employee("Harsh Patel"))            # one employee
# s.get_balance("Harsh Patel")             # balance number

['Piyush Sureshbhai Kanani', 'Nikhilesh Ramoliya', 'Shivani', 'Vipul Lakhara', 'Harsh Patel', 'Akhil Shah', 'Meet Patel', 'Nidhi Patel', 'Dhvanika Monapara', 'Dipesh Sojitra', 'Zil Hingu', 'Het Mewada', 'Yogesh Patel', 'Yash Dudhat']
{'id': '55', 'leaves': {'1 Mar 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Apr 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 May 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Jun 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Jul 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Aug 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Sep 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Oct 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Nov 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Dec 2022': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Jan 2023': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Feb 2023': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '1 Mar 2023': {'credit': 0.0, 'lwp': 0.0, 'debit': 0.0}, '